# Exploration des données de barrages

Ce notebook explore les données nettoyées (réelles ou synthétiques) selon la configuration `DATA_SOURCE`.

In [1]:
import os
import sys
sys.path.insert(0, '/home/tawil/projets/barrages-maroc')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

from src.config import DATA_PROCESSED, DATA_SOURCE

print(f"DATA_SOURCE = {DATA_SOURCE}")

DATA_SOURCE = synthetic


In [2]:
# 1. Charge le parquet nettoyé selon DATA_SOURCE
if DATA_SOURCE == "real":
    path = DATA_PROCESSED / "barrages_clean.parquet"
else:
    path = DATA_PROCESSED / "synthetic" / "barrages_clean_synthetic.parquet"

print(f"Chargement depuis : {path}")
df = pd.read_parquet(path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["barrage", "date"]).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Colonnes: {df.columns.tolist()}")
print(f"Période: {df['date'].min()} à {df['date'].max()}")
print(f"Barrages uniques: {df['barrage'].nunique()}")
print(f"Source: {df['source'].iloc[0] if 'source' in df.columns else 'N/A'}")
df.head()

Chargement depuis : /home/tawil/projets/barrages-maroc/data/processed/synthetic/barrages_clean_synthetic.parquet
Shape: (32400, 12)
Colonnes: ['date', 'barrage', 'niveau_pct', 'source', 'aberration_flag', 'nom', 'bassin_versant', 'latitude', 'longitude', 'capacite_max_m3', 'hauteur_m', 'annee_mise_en_service']
Période: 2000-01-01 00:00:00 à 2026-12-01 00:00:00
Barrages uniques: 100
Source: synthetic


,date,barrage,niveau_pct,source,aberration_flag,nom,bassin_versant,latitude,longitude,capacite_max_m3,hauteur_m,annee_mise_en_service
0,2000-01-01,generic_Draa_55,100.00,synthetic,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,generic_Draa_55,80.83,synthetic,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-03-01,generic_Draa_55,71.69,synthetic,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-04-01,generic_Draa_55,28.12,synthetic,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-05-01,generic_Draa_55,16.59,synthetic,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# 2. Graphique plotly : évolution du niveau par série
fig = px.line(
    df,
    x="date",
    y="niveau_pct",
    color="barrage",
    title=f"Évolution du niveau de remplissage ({DATA_SOURCE})",
    labels={"niveau_pct": "Niveau (%)", "date": "Date", "barrage": "Barrage"},
    height=600
)
fig.update_layout(
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

In [4]:
# 3. Statistiques descriptives
print("=" * 60)
print(f"STATISTIQUES GLOBALES ({DATA_SOURCE})")
print("=" * 60)
print(f"Période couverte: {df['date'].min().date()} à {df['date'].max().date()}")
print(f"Nombre d'observations: {len(df):,}")
print(f"Nombre de barrages/séries: {df['barrage'].nunique()}")
print(f"Nombre d'années: {(df['date'].max() - df['date'].min()).days / 365.25:.1f}")
print(f"% manquants (niveau_pct): {df['niveau_pct'].isna().mean() * 100:.2f}%")
print(f"Niveau min: {df['niveau_pct'].min():.1f}%")
print(f"Niveau max: {df['niveau_pct'].max():.1f}%")
print(f"Niveau moyen: {df['niveau_pct'].mean():.1f}%")
print(f"Niveau médian: {df['niveau_pct'].median():.1f}%")

# Saisonnalité mensuelle moyenne
df['month'] = df['date'].dt.month
monthly_avg = df.groupby('month')['niveau_pct'].mean()
print("\nSaisonnalité mensuelle moyenne (niveau_pct):")
for m, v in monthly_avg.items():
    print(f"  Mois {m:2d}: {v:.1f}%")

# Amplitude saisonnière
print(f"\nAmplitude saisonnière (max - min mensuel): {monthly_avg.max() - monthly_avg.min():.1f}%")

STATISTIQUES GLOBALES (synthetic)
Période couverte: 2000-01-01 à 2026-12-01
Nombre d'observations: 32,400
Nombre de barrages/séries: 100
Nombre d'années: 26.9
% manquants (niveau_pct): 0.00%
Niveau min: 0.0%
Niveau max: 100.0%
Niveau moyen: 55.1%
Niveau médian: 54.7%

Saisonnalité mensuelle moyenne (niveau_pct):
  Mois  1: 89.3%
  Mois  2: 90.4%
  Mois  3: 88.6%
  Mois  4: 70.2%
  Mois  5: 52.8%
  Mois  6: 17.3%
  Mois  7: 20.1%
  Mois  8: 14.7%
  Mois  9: 19.8%
  Mois 10: 45.4%
  Mois 11: 69.9%
  Mois 12: 83.1%

Amplitude saisonnière (max - min mensuel): 75.7%


In [5]:
# 4. Tableau des caractéristiques des barrages
if 'capacite_max_m3' in df.columns:
    dam_info = df.groupby('barrage').agg({
        'bassin_versant': 'first',
        'latitude': 'first',
        'longitude': 'first',
        'capacite_max_m3': 'first',
        'hauteur_m': 'first',
        'annee_mise_en_service': 'first',
        'niveau_pct': ['mean', 'min', 'max'],
        'date': ['min', 'max', 'count']
    }).round(2)
    dam_info.columns = ['bassin', 'lat', 'lon', 'capacite_Mm3', 'hauteur_m', 'annee_service',
                        'niveau_moyen', 'niveau_min', 'niveau_max',
                        'date_debut', 'date_fin', 'nb_obs']
    dam_info['capacite_Mm3'] = (dam_info['capacite_Mm3'] / 1e6).round(1)
    print(dam_info.to_string())
else:
    print("Pas de caractéristiques statiques disponibles (données synthétiques génériques)")
    # Pour synthétique : résumé par scénario
    if 'scenario' in df.columns:
        scenario_info = df.groupby('scenario').agg({
            'niveau_pct': ['mean', 'min', 'max', 'std'],
            'date': ['min', 'max', 'count'],
            'barrage': 'nunique'
        }).round(2)
        scenario_info.columns = ['niveau_moyen', 'niveau_min', 'niveau_max', 'niveau_std',
                               'date_debut', 'date_fin', 'nb_obs', 'nb_series']
        print("\nRésumé par scénario:")
        print(scenario_info.to_string())

                       bassin  lat  lon  capacite_Mm3  hauteur_m  annee_service  niveau_moyen  niveau_min  niveau_max date_debut   date_fin  nb_obs
barrage                                                                                                                                            
generic_Draa_55           NaN  NaN  NaN           NaN        NaN            NaN         34.64        0.00       100.0 2000-01-01 2026-12-01     324
generic_Draa_56           NaN  NaN  NaN           NaN        NaN            NaN         38.67        0.00       100.0 2000-01-01 2026-12-01     324
generic_Draa_57           NaN  NaN  NaN           NaN        NaN            NaN         35.28        0.00       100.0 2000-01-01 2026-12-01     324
generic_Draa_58           NaN  NaN  NaN           NaN        NaN            NaN         38.10        0.00       100.0 2000-01-01 2026-12-01     324
generic_Draa_59           NaN  NaN  NaN           NaN        NaN            NaN         38.45        0.00       

/tmp/ipykernel_50290/2827286141.py:12: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  }).round(2)


In [6]:
# 5. Analyse des sécheresses (périodes basses prolongées)
print("ANALYSE DES SÈCHERESSES")
print("=" * 40)

drought_counts = {}
for barrage in df['barrage'].unique():
    bdf = df[df['barrage'] == barrage].sort_values('date')
    low_threshold = bdf['niveau_pct'].quantile(0.25)
    below = (bdf['niveau_pct'] < low_threshold).astype(int)
    
    max_streak = 0
    current = 0
    streaks = []
    for v in below:
        if v == 1:
            current += 1
        else:
            if current > 0:
                streaks.append(current)
            current = 0
    if current > 0:
        streaks.append(current)
    
    max_streak = max(streaks) if streaks else 0
    drought_counts[barrage] = {
        'max_streak_months': max_streak,
        'low_threshold': low_threshold,
        'num_droughts_6m': sum(1 for s in streaks if s >= 6)
    }

drought_df = pd.DataFrame(drought_counts).T.sort_values('max_streak_months', ascending=False)
print(drought_df.head(10).to_string())
print(f"\nSéries avec sécheresse >= 6 mois: {(drought_df['num_droughts_6m'] > 0).sum()} / {len(drought_df)}")
print(f"Durée max de sécheresse observée: {drought_df['max_streak_months'].max()} mois")

ANALYSE DES SÈCHERESSES
                        max_streak_months  low_threshold  num_droughts_6m
generic_Draa_58                       6.0         4.6300              2.0
generic_Inaouen_26                    6.0        11.9700              1.0
generic_Inaouen_29                    6.0         9.7350              1.0
generic_Draa_65                       6.0         3.6075              1.0
generic_Inaouen_31                    6.0        10.1600              1.0
generic_Inaouen_30                    6.0        11.5150              1.0
generic_Oum Er-Rbia_7                 6.0        23.4025              1.0
generic_Oum Er-Rbia_23                6.0        20.1625              1.0
generic_Oum Er-Rbia_21                6.0        21.4125              1.0
generic_Oum Er-Rbia_6                 6.0        21.0100              1.0

Séries avec sécheresse >= 6 mois: 13 / 100
Durée max de sécheresse observée: 6.0 mois


# VERDICT DONNÉES

---

**Source des données :** `{{DATA_SOURCE}}`

**Années par série :** `{{N_YEARS}}` ans ({{N_MONTHS}} mois par série)

**Saisonnalité visible :** `{{SEASONALITY}}`

**Sécheresses visibles :** `{{DROUGHTS}}`

**Qualité globale :** `{{QUALITY}}`

---

*Remplacez les variables entre {{ }} après exécution du notebook.*